# PRS and NMR C-index comparison

Out-of-fold C-index for BASE, CLIN, and SOC models, each compared with PRS, NMR, and PRS+NMR.
Models are the 12 `{prefix}{base|clin|marker}[_prs][_nmr]_oof.rds` files in `outputs/main/`.


In [ ]:
library(tidyverse)
library(patchwork)
library(here)

source(here("helpers/oof_prs_cindex.r"))

fig_dir <- here("outputs/figures")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)
oof_dir <- here("outputs/main")


In [ ]:
disease_specs <- list(
  t2d = list(
    title = "T2D", long = "Type 2 Diabetes",
    prefix = "t2d_a1c_", primary = TRUE
  ),
  ckd = list(
    title = "CKD", long = "Chronic Kidney Disease",
    prefix = "ckd_egfr_", primary = TRUE
  ),
  masld = list(
    title = "MASLD", long = "Metabolic Dysfunctional Steatotic Liver Disease",
    prefix = "nafld_fib4_", primary = TRUE
  ),
  hf_cm = list(
    title = "Heart Failure", long = "Heart Failure",
    prefix = "hf_cm_prevent_ingredients_", primary = TRUE
  ),
  acute_mi = list(
    title = "Acute MI", long = "Acute MI",
    prefix = "acute_mi_prevent_ingredients_", primary = TRUE
  ),
  total_cvd = list(
    title = "Total CVD", long = "Total CVD",
    prefix = "total_cvd_prevent_ingredients_", primary = TRUE
  ),
  chd = list(
    title = "CHD", long = "Coronary Heart Disease",
    prefix = "chd_revasc_prevent_ingredients_", primary = TRUE
  ),
  tia_stroke = list(
    title = "TIA/Stroke", long = "TIA/Stroke",
    prefix = "tia_stroke_prevent_ingredients_", primary = TRUE
  ),
  afib = list(
    title = "Afib", long = "Atrial Fibrillation",
    prefix = "afib_chargeaf_ingredients_", primary = TRUE
  )
)

primary_ids <- names(Filter(function(spec) spec$primary, disease_specs))

cindex <- lapply(disease_specs, function(spec) {
  collect_prs_cindex(oof_dir, spec$prefix)
})


In [ ]:
xlim <- c(0.5, 0.85)

p_prs <- patchwork::wrap_plots(
  lapply(primary_ids, function(id) {
    plot_oof_prs_nmr_cindex(cindex[[id]], disease = disease_specs[[id]]$long, xlim = xlim) +
      theme(legend.position = "none")
  }),
  ncol = 2
)

options(repr.plot.width = 10, repr.plot.height = 7)
p_prs

ggsave(file.path(fig_dir, "PRS_FIG.png"), p_prs, width = 10, height = 7, dpi = 600)


In [ ]:
col_order <- c(
  "endpoint",
  "BASE", "BASE+PRS", "BASE+NMR", "BASE+PRS+NMR",
  "CLIN", "CLIN+PRS", "CLIN+NMR", "CLIN+PRS+NMR",
  "SOC", "SOC+PRS", "SOC+NMR", "SOC+PRS+NMR"
)

cindex_prs_table <- bind_rows(lapply(primary_ids, function(id) {
  cindex[[id]] %>% mutate(endpoint = disease_specs[[id]]$title)
})) %>%
  mutate(
    model = recode(as.character(model), base = "BASE", clin = "CLIN", marker = "SOC"),
    col = case_when(
      type == "SOC"         ~ model,
      type == "SOC+PRS"     ~ paste0(model, "+PRS"),
      type == "SOC+NMR"     ~ paste0(model, "+NMR"),
      type == "SOC+PRS+NMR" ~ paste0(model, "+PRS+NMR")
    ),
    value = sprintf("%.3f (%.3f, %.3f)", c_index, ci_lower, ci_upper),
    endpoint = factor(endpoint, levels = vapply(disease_specs[primary_ids], `[[`, "", "title"))
  ) %>%
  select(endpoint, col, value) %>%
  pivot_wider(names_from = col, values_from = value) %>%
  arrange(endpoint) %>%
  select(all_of(col_order))

cindex_prs_table

table_dir <- here("outputs/tables")
dir.create(table_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(cindex_prs_table, file.path(table_dir, "CINDEX_PRS_STATS.csv"), row.names = FALSE)